# Data Analytics Project — Project Cheat Sheet

## Purpose

This notebook is the project's quick-reference guide for the commands, SQL queries, validation checks, data-export commands, troubleshooting steps, and Git workflow used throughout the project.

**Project:** `data_analytics`  
**Database:** `daimonupdown`  
**Shell:** Bash / Arch Linux terminal  
**Database:** MariaDB  
**BI:** Tableau Public  
**Version Control:** Git / GitHub  

Use this notebook as a command reference. Detailed dashboard-specific workflows are documented separately in the Workforce and Employee Retention notebooks.

**Important:** SQL commands must be executed through MariaDB. Bash and Git commands are executed at the terminal prompt.

## 1. Project Process

The general project workflow is:

**MariaDB / SQL → validation → data preparation → CSV → Tableau → Git/GitHub**

Dashboard-specific workflows are documented in:

- `03_workforce_dashboard.ipynb`
- `04_employee_retention_dashboard.ipynb`

## 2. Git / GitHub Commands

In [ ]:
# Current Git status
git status

# See every tracked file
git ls-files

# See files tracked in a specific folder
git ls-files sql/
git ls-files python/
git ls-files notebooks/

# Check GitHub remote
git remote -v

# See recent commits
git log --oneline --decorate -10

# Refresh remote information
git fetch origin

# Show files actually tracked on origin/main
git ls-tree -r --name-only origin/main | sort

# Stage changes
git add <file-or-folder>

# Commit changes
git commit -m "Describe the change"

# Push to GitHub
git push

# Final verification
git status

## 3. MariaDB / SQL Basics

In [ ]:
# Check whether LOCAL INFILE is enabled
sudo mariadb -e "SHOW VARIABLES LIKE 'local_infile';"

# Connect to the database and show tables
sudo mariadb -e "USE daimonupdown; SHOW TABLES;"

# Describe a table
sudo mariadb -e "USE daimonupdown; DESCRIBE employees;"

# Run SQL directly from the terminal
sudo mariadb -e "USE daimonupdown; SELECT COUNT(*) FROM employees;"

# Important:
# SQL such as SELECT, SET, SHOW, DESCRIBE, etc. is not Bash.
# Do not type SQL directly at the [lab@archlabi data_analytics]$ prompt.

## 4. Run a SQL File

In [ ]:
# Run a SQL script against MariaDB
sudo mariadb --local-infile=1 < sql/03_load_data.sql

# If the script explicitly needs the database name
sudo mariadb --local-infile=1 daimonupdown < sql/03_load_data.sql

## 5. LOAD DATA LOCAL INFILE

In [ ]:
sudo mariadb --local-infile=1 -e "
USE daimonupdown;
LOAD DATA LOCAL INFILE '/home/lab/Desktop/GitHub/data_analytics/data/raw/departments.csv'
INTO TABLE departments
FIELDS TERMINATED BY ','
ENCLOSED BY '\"'
LINES TERMINATED BY '\n'
IGNORE 1 LINES;
SELECT COUNT(*) AS department_count FROM departments;
"

## 6. Controlled Reloads / Foreign-Key Checks

In [ ]:
# SQL must be passed to MariaDB.
sudo mariadb -e "
USE daimonupdown;
SET FOREIGN_KEY_CHECKS=0;
DELETE FROM clients;
SET FOREIGN_KEY_CHECKS=1;
"

## 7. Data Validation Queries

These checks are used to verify that the database contains usable data before dashboard development.

### 7.1 Row Counts

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT 'applications' AS table_name, COUNT(*) AS row_count FROM applications
UNION ALL SELECT 'assignments', COUNT(*) FROM assignments
UNION ALL SELECT 'attendance', COUNT(*) FROM attendance
UNION ALL SELECT 'candidates', COUNT(*) FROM candidates
UNION ALL SELECT 'client_feedback', COUNT(*) FROM client_feedback
UNION ALL SELECT 'clients', COUNT(*) FROM clients
UNION ALL SELECT 'compensation_history', COUNT(*) FROM compensation_history
UNION ALL SELECT 'departments', COUNT(*) FROM departments
UNION ALL SELECT 'employees', COUNT(*) FROM employees
UNION ALL SELECT 'employee_surveys', COUNT(*) FROM employee_surveys
UNION ALL SELECT 'employment_history', COUNT(*) FROM employment_history
UNION ALL SELECT 'locations', COUNT(*) FROM locations
UNION ALL SELECT 'offboarding', COUNT(*) FROM offboarding
UNION ALL SELECT 'onboarding', COUNT(*) FROM onboarding
UNION ALL SELECT 'performance', COUNT(*) FROM performance
UNION ALL SELECT 'positions', COUNT(*) FROM positions
UNION ALL SELECT 'qualifications', COUNT(*) FROM qualifications
UNION ALL SELECT 'schools', COUNT(*) FROM schools
UNION ALL SELECT 'training', COUNT(*) FROM training
UNION ALL SELECT 'visa_history', COUNT(*) FROM visa_history;
"

### 7.2 Foreign-Key Relationships

In [ ]:
sudo mariadb -e "
SELECT
    TABLE_NAME,
    COLUMN_NAME,
    REFERENCED_TABLE_NAME,
    REFERENCED_COLUMN_NAME,
    CONSTRAINT_NAME
FROM INFORMATION_SCHEMA.KEY_COLUMN_USAGE
WHERE TABLE_SCHEMA = 'daimonupdown'
  AND REFERENCED_TABLE_NAME IS NOT NULL
ORDER BY TABLE_NAME, COLUMN_NAME;
"

### 7.3 Orphan Records

In [ ]:
sudo mariadb -e "
USE daimonupdown;

SELECT COUNT(*) AS orphaned_assignments
FROM assignments a
LEFT JOIN employees e ON a.employee_id = e.employee_id
WHERE e.employee_id IS NULL;

SELECT COUNT(*) AS orphaned_training
FROM training t
LEFT JOIN employees e ON t.employee_id = e.employee_id
WHERE e.employee_id IS NULL;

SELECT COUNT(*) AS orphaned_performance
FROM performance p
LEFT JOIN employees e ON p.employee_id = e.employee_id
WHERE e.employee_id IS NULL;

SELECT COUNT(*) AS orphaned_attendance
FROM attendance a
LEFT JOIN employees e ON a.employee_id = e.employee_id
WHERE e.employee_id IS NULL;
"

### 7.4 Duplicate Primary Keys

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT
    'employees' AS table_name,
    COUNT(*) - COUNT(DISTINCT employee_id) AS duplicate_ids
FROM employees;
"

### 7.5 Invalid Date Relationships

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT COUNT(*) AS invalid_client_dates
FROM clients
WHERE client_relationship_end_date IS NOT NULL
  AND client_relationship_start_date > client_relationship_end_date;

SELECT COUNT(*) AS invalid_assignment_dates
FROM assignments
WHERE assignment_end_date IS NOT NULL
  AND assignment_start_date > assignment_end_date;

SELECT COUNT(*) AS invalid_employment_dates
FROM employment_history
WHERE employment_end_date IS NOT NULL
  AND employment_start_date > employment_end_date;

SELECT COUNT(*) AS invalid_onboarding_dates
FROM onboarding
WHERE onboarding_end_date IS NOT NULL
  AND onboarding_start_date > onboarding_end_date;
"

### 7.6 NULL Checks

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT 'employees.employee_id' AS field_name, COUNT(*) - COUNT(employee_id) AS null_count FROM employees
UNION ALL SELECT 'employees.candidate_id', COUNT(*) - COUNT(candidate_id) FROM employees
UNION ALL SELECT 'assignments.employee_id', COUNT(*) - COUNT(employee_id) FROM assignments
UNION ALL SELECT 'assignments.client_id', COUNT(*) - COUNT(client_id) FROM assignments
UNION ALL SELECT 'performance.employee_id', COUNT(*) - COUNT(employee_id) FROM performance
UNION ALL SELECT 'attendance.employee_id', COUNT(*) - COUNT(employee_id) FROM attendance;
"

### 7.7 Category Values

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT DISTINCT client_status FROM clients ORDER BY client_status;
SELECT DISTINCT assignment_status FROM assignments ORDER BY assignment_status;
SELECT DISTINCT employment_status FROM employment_history ORDER BY employment_status;
SELECT DISTINCT candidate_status FROM candidates ORDER BY candidate_status;
SELECT DISTINCT completion_status FROM training ORDER BY completion_status;
"

## 8. Analysis Query Patterns

### 8.1 Recruitment Analysis

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT
    application_status,
    COUNT(*) AS applications
FROM applications
GROUP BY application_status
ORDER BY applications DESC;
"

# Hired applications vs distinct hired candidates
sudo mariadb -e "
USE daimonupdown;
SELECT
    COUNT(*) AS hired_applications,
    COUNT(DISTINCT candidate_id) AS hired_candidates
FROM applications
WHERE application_status = 'Hired';
"

# Application hire rate
sudo mariadb -e "
USE daimonupdown;
SELECT
    COUNT(*) AS total_applications,
    SUM(application_status = 'Hired') AS hired_applications,
    COUNT(DISTINCT CASE WHEN application_status = 'Hired' THEN candidate_id END) AS hired_candidates,
    ROUND(100.0 * SUM(application_status = 'Hired') / COUNT(*), 2) AS application_hire_rate
FROM applications;
"

### 8.2 Repeat Hired Candidates

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT
    candidate_id,
    COUNT(*) AS hired_application_count
FROM applications
WHERE application_status = 'Hired'
GROUP BY candidate_id
HAVING COUNT(*) > 1
ORDER BY hired_application_count DESC;
"

### 8.3 Recruitment Source Performance

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT
    c.recruitment_source,
    COUNT(DISTINCT c.candidate_id) AS candidates,
    COUNT(DISTINCT CASE WHEN a.application_status = 'Hired' THEN a.candidate_id END) AS hired_candidates,
    ROUND(
        100.0 * COUNT(DISTINCT CASE WHEN a.application_status = 'Hired' THEN a.candidate_id END)
        / COUNT(DISTINCT c.candidate_id), 2
    ) AS candidate_hire_rate
FROM candidates c
LEFT JOIN applications a ON c.candidate_id = a.candidate_id
GROUP BY c.recruitment_source
ORDER BY candidate_hire_rate DESC;
"

### 8.4 Position Performance

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT
    position_applied,
    COUNT(*) AS applications,
    SUM(application_status = 'Hired') AS hired_applications,
    COUNT(DISTINCT CASE WHEN application_status = 'Hired' THEN candidate_id END) AS hired_candidates,
    ROUND(100.0 * SUM(application_status = 'Hired') / COUNT(*), 2) AS application_hire_rate
FROM applications
GROUP BY position_applied
ORDER BY applications DESC;
"

## 9. Export Data for Tableau

In [ ]:
# Create dashboard data folder
mkdir -p dashboard/data

# Export with headers. Do NOT use --skip-column-names.
sudo mariadb --batch -e "USE daimonupdown; SELECT * FROM employees;" > dashboard/data/employees.tsv
sudo mariadb --batch -e "USE daimonupdown; SELECT * FROM assignments;" > dashboard/data/assignments.tsv
sudo mariadb --batch -e "USE daimonupdown; SELECT * FROM clients;" > dashboard/data/clients.tsv
sudo mariadb --batch -e "USE daimonupdown; SELECT * FROM attendance;" > dashboard/data/attendance.tsv

# Convert TSV files to CSV
python python/tsv_to_csv.py dashboard/data/

# Remove intermediate TSV files
rm dashboard/data/*.tsv

## 10. Workforce Tableau Dataset

In [ ]:
# Activate the project environment when needed
source .venv/bin/activate

# Build the Tableau-ready workforce dataset
python python/create_workforce_dashboard.py

The Workforce Dashboard dataset was verified at **2,341 rows and 40 columns**, with one row representing an assignment.

The Workforce Dashboard workflow itself uses MariaDB/SQL, terminal commands, CSV output, and Tableau. Python utilities are documented here as reusable project tools.

## 11. Python Utility Commands

In [ ]:
# Convert one TSV file to CSV
python python/tsv_to_csv.py input.tsv

# Convert every TSV in a folder
python python/tsv_to_csv.py input_folder/ output_folder/

# Merge Excel workbooks in a folder
python python/merge_excel_to_csv.py excel_folder csv_folder

## 12. MariaDB Backup

In [ ]:
# Complete database dump
sudo mariadb-dump --databases daimonupdown > sql/daimonupdown_dump.sql

# Check dump size
ls -lh sql/daimonupdown_dump.sql

# Check beginning of dump
head -n 20 sql/daimonupdown_dump.sql

## 13. Troubleshooting / Common Mistakes

### SQL typed at the Bash prompt

Incorrect:

```text
[lab@archlabi data_analytics]$ SELECT COUNT(*) FROM departments;
bash: syntax error
```

Correct:

```text
sudo mariadb -e "SELECT COUNT(*) FROM departments;"
```
### Running a SQL filename as a command

Incorrect:

```text
sql/03_load_data.sql
```
Correct:

```text
sudo mariadb --local-infile=1 < sql/03_load_data.sql
```
### MariaDB is not running

Check:

```text
sudo systemctl status mariadb
```
Start when necessary:

```text
sudo systemctl start mariadb
```
### LOCAL INFILE error

Use `--local-infile=1` on the MariaDB client when loading local files.

### TSV export without headers

Do not use `--skip-column-names` when the exported file will be converted to CSV for Tableau.

### Git workflow

After making changes:

**`git status` → `git add` → `git commit` → `git push` → `git status`**

## 14. Project Notebook Structure

```text
notebooks/
├── 01_project_cheat_sheet.ipynb
├── 02_data_cleaning.ipynb
├── 03_workforce_dashboard.ipynb
└── 04_employee_retention_dashboard.ipynb
```

### Notebook roles

**01 — Project Cheat Sheet**  
Reusable commands, SQL queries, validation checks, troubleshooting, and project references.

**02 — Data Cleaning**  
Detailed database/data-quality preparation and validation work.

**03 — Workforce Dashboard**  
Assignment-level workforce analysis and Tableau dashboard development.

**04 — Employee Retention Dashboard**  
Employee-level employment, retention, turnover, and offboarding analysis.

## 15. Final Working Principle

Keep the command history and SQL queries that were actually used.

Do not claim that a tool or process was used if it was not part of the actual workflow.

Use the dashboard-specific notebooks to document the exact steps, decisions, troubleshooting, and results for each dashboard.